In [ ]:
PD and NPD caculation and save images

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import cv2
import tifffile
from skimage.feature import greycomatrix, greycoprops
from tqdm import tqdm
from sklearn.preprocessing import RobustScaler

# -----------------------------
# 1. Configuration of Parameters and Paths
# -----------------------------
data_dirs = {
    "Control": r"E:\0. Research\1. Paper_data\2. Hacat cisplatin ROS\Original\Control",
    "CIS":     r"E:\0. Research\1. Paper_data\2. Hacat cisplatin ROS\Original\CIS",
    "HIS":     r"E:\0. Research\1. Paper_data\2. Hacat cisplatin ROS\Original\HIS"
}

# Root output directory
base_output_dir = r"E:\0. Research\1. Paper_data\2. Hacat cisplatin ROS\Original\Polarization_Diff_Features"
os.makedirs(base_output_dir, exist_ok=True)

# Subfolders: Save differential images
img_out_dirs = {
    "PD": os.path.join(base_output_dir, "Images", "PD"),
    "NPD": os.path.join(base_output_dir, "Images", "NPD")
}
for d in img_out_dirs.values():
    os.makedirs(d, exist_ok=True)

# Subfolders: Save CSV features
csv_out_dir = os.path.join(base_output_dir, "CSV")
os.makedirs(csv_out_dir, exist_ok=True)

resize_size = 128

angles = [0, np.pi/4, np.pi/2, 3*np.pi/4]
angle_labels = {
    0: "0",
    np.pi/4: "45",
    np.pi/2: "90",
    3*np.pi/4: "135"
}

distances = [1, 2, 4, 8, 16, 32, 64]
properties = ['ASM', 'contrast', 'correlation', 'homogeneity', 'dissimilarity']

# -----------------------------
# 2. Dynamically Generate Descriptive Feature Column Names
# -----------------------------
def generate_feature_names(diff_type):
    """
    Generate column names following the flattening order of skimage.feature.greycoprops().flatten()
    Order: outer loop properties -> inner flatten (first distances, then angles)
    E.g.: contrast_d=16_theta=45_PD
    """
    feat_names = []
    for prop in properties:
        for d in distances:
            for a in angles:
                theta = angle_labels[a]
                name = f"{prop}_d={d}_theta={theta}_{diff_type}"
                feat_names.append(name)
    return feat_names

# Generate 140-dimensional parameterized feature names for PD and NPD respectively
pd_feature_columns = generate_feature_names("PD")
npd_feature_columns = generate_feature_names("NPD")

# -----------------------------
# 3. Helper Function Definitions
# -----------------------------
def get_image_pairs(folder):
    """Match P/S image pairs"""
    p_files = sorted(glob.glob(os.path.join(folder, "*-P*.tif*")))
    pairs = []
    for p_file in p_files:
        s_file = p_file.replace("-P", "-S")
        if os.path.exists(s_file):
            pairs.append((p_file, s_file))
    return pairs

def extract_glcm_features(img):
    """Extract GLCM features for a single image (140 dimensions)"""
    img_uint8 = (img * 255).astype(np.uint8)
    glcm = greycomatrix(img_uint8, distances=distances, angles=angles, symmetric=True, normed=True)
    feats = []
    for prop in properties:
        feats.extend(greycoprops(glcm, prop).flatten())
    return feats

# -----------------------------
# 4. Collect All Image Pairs
# -----------------------------
all_pairs = []
for label, folder in data_dirs.items():
    pairs = get_image_pairs(folder)
    print(f"Found {len(pairs)} image pairs in folder [{label}]")
    for p_file, s_file in pairs:
        all_pairs.append((p_file, s_file, label))

print(f"\nTotal number of collected image pairs: {len(all_pairs)}")

pd_results = []
npd_results = []

# -----------------------------
# 5. Process Iteratively, Save Images, and Extract GLCM Features
# -----------------------------
for p_file, s_file, label in tqdm(all_pairs, desc="Processing Pairs & Extracting GLCM"):
    try:
        # Read, resize to 128x128, and normalize to 0~1 float
        p_img = cv2.resize(tifffile.imread(p_file).astype(np.float32)/65535., (resize_size, resize_size))
        s_img = cv2.resize(tifffile.imread(s_file).astype(np.float32)/65535., (resize_size, resize_size))
        
        base_name = os.path.basename(p_file)
        
        # -----------------------------
        # 1. Polarization Difference PD (with absolute value): |P - S|
        # -----------------------------
        pd_img = np.abs(p_img - s_img)
        pd_norm = (pd_img - pd_img.min()) / (pd_img.max() - pd_img.min() + 1e-8)
        pd_16 = (pd_norm * 65535).astype(np.uint16)
        
        label_pd_dir = os.path.join(img_out_dirs["PD"], label)
        os.makedirs(label_pd_dir, exist_ok=True)
        pd_save_path = os.path.join(label_pd_dir, base_name)
        tifffile.imwrite(pd_save_path, pd_16)
        
        # -----------------------------
        # 2. Normalized Polarization Difference NPD: (P - S) / (P + S)
        # -----------------------------
        npd_img = (p_img - s_img) / (p_img + s_img + 1e-8)
        npd_norm = (npd_img - npd_img.min()) / (npd_img.max() - npd_img.min() + 1e-8)
        npd_16 = (npd_norm * 65535).astype(np.uint16)
        
        label_npd_dir = os.path.join(img_out_dirs["NPD"], label)
        os.makedirs(label_npd_dir, exist_ok=True)
        npd_save_path = os.path.join(label_npd_dir, base_name)
        tifffile.imwrite(npd_save_path, npd_16)
        
        # -----------------------------
        # 3. Extract GLCM Features (140 dims each)
        # -----------------------------
        pd_feats = extract_glcm_features(pd_norm)
        npd_feats = extract_glcm_features(npd_norm)
        
        pd_results.append([label, p_file, s_file, pd_save_path] + pd_feats)
        npd_results.append([label, p_file, s_file, npd_save_path] + npd_feats)
        
    except Exception as e:
        print(f"Error processing {p_file} / {s_file}: {e}")

# -----------------------------
# 6. Save Raw Features and RobustScaler Normalized Feature CSV Files
# -----------------------------
meta_columns = ['Label', 'P_Path', 'S_Path', 'Diff_Path']

# --- PD Feature Processing ---
df_pd = pd.DataFrame(pd_results, columns=meta_columns + pd_feature_columns)
pd_raw_csv_path = os.path.join(csv_out_dir, "pd_glcm_features_named.csv")
df_pd.to_csv(pd_raw_csv_path, index=False)

# Normalize PD features using RobustScaler
scaler_pd = RobustScaler()
df_pd_scaled = df_pd.copy()
df_pd_scaled[pd_feature_columns] = scaler_pd.fit_transform(df_pd_scaled[pd_feature_columns])
pd_scaled_csv_path = os.path.join(csv_out_dir, "pd_glcm_features_named_robust_scaled.csv")
df_pd_scaled.to_csv(pd_scaled_csv_path, index=False)

# --- NPD Feature Processing ---
df_npd = pd.DataFrame(npd_results, columns=meta_columns + npd_feature_columns)
npd_raw_csv_path = os.path.join(csv_out_dir, "npd_glcm_features_named.csv")
df_npd.to_csv(npd_raw_csv_path, index=False)

# Normalize NPD features using RobustScaler
scaler_npd = RobustScaler()
df_npd_scaled = df_npd.copy()
df_npd_scaled[npd_feature_columns] = scaler_npd.fit_transform(df_npd_scaled[npd_feature_columns])
npd_scaled_csv_path = os.path.join(csv_out_dir, "npd_glcm_features_named_robust_scaled.csv")
df_npd_scaled.to_csv(npd_scaled_csv_path, index=False)

print(f"\n✅ Raw features and RobustScaler normalized named feature CSVs have been successfully saved to: {csv_out_dir}")

In [ ]:
classification

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import cv2
import tifffile
from skimage.feature import greycomatrix, greycoprops
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import RobustScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# -----------------------------
# 1. Configuration of Parameters and Paths
# -----------------------------
data_dirs = {
    "Control": r"E:\0. Research\1. Paper_data\2. Hacat cisplatin ROS\Original\Control",
    "CIS":     r"E:\0. Research\1. Paper_data\2. Hacat cisplatin ROS\Original\CIS",
    "HIS":     r"E:\0. Research\1. Paper_data\2. Hacat cisplatin ROS\Original\HIS"
}

base_output_dir = r"E:\0. Research\1. Paper_data\2. Hacat cisplatin ROS\Original\差分TestAll"
os.makedirs(base_output_dir, exist_ok=True)

# Subfolders: Save images
img_out_dirs = {
    "PD": os.path.join(base_output_dir, "Images", "PD"),
    "NPD": os.path.join(base_output_dir, "Images", "NPD")
}
for d in img_out_dirs.values():
    os.makedirs(d, exist_ok=True)

# Subfolders: Save CSV
csv_out_dir = os.path.join(base_output_dir, "CSV")
os.makedirs(csv_out_dir, exist_ok=True)

resize_size = 128 
random_seed = 42

angles = [0, np.pi/4, np.pi/2, 3*np.pi/4]
angle_labels = {
    0: "0",
    np.pi/4: "45",
    np.pi/2: "90",
    3*np.pi/4: "135"
}
distances = [1, 2, 4, 8, 16, 32, 64]
properties = ['ASM', 'contrast', 'correlation', 'homogeneity', 'dissimilarity']

# -----------------------------
# 2. Helper Function Definitions
# -----------------------------
def get_image_pairs(folder):
    """Match P/S image pairs"""
    p_files = sorted(glob.glob(os.path.join(folder, "*-P*.tif*")))
    pairs = []
    for p_file in p_files:
        s_file = p_file.replace("-P", "-S")
        if os.path.exists(s_file):
            pairs.append((p_file, s_file))
    return pairs

def extract_glcm_features(img):
    """Extract single image GLCM features (140-dimensional)"""
    img_uint8 = (img * 255).astype(np.uint8)
    glcm = greycomatrix(img_uint8, distances=distances, angles=angles, symmetric=True, normed=True)
    feats = []
    for prop in properties:
        feats.extend(greycoprops(glcm, prop).flatten())
    return feats

def generate_feature_names(channel_name):
    """
    Generate column names following the flattening order of skimage.feature.greycoprops().flatten()
    Order: outer properties -> middle distances -> inner angles
    E.g.: contrast_d=1_theta=45_PD
    """
    feat_names = []
    for prop in properties:
        for d in distances:
            for a in angles:
                theta = angle_labels[a]
                name = f"{prop}_d={d}_theta={theta}_{channel_name}"
                feat_names.append(name)
    return feat_names

def process_single_pair(item, pd_limit):
    """Single image pair processing function (multithreading safe, uses fixed pd_limit)"""
    p_file, s_file, label = item
    try:
        # Read original resolution images
        p_img = tifffile.imread(p_file).astype(np.float32) / 65535.0
        s_img = tifffile.imread(s_file).astype(np.float32) / 65535.0
        
        base_name = os.path.basename(p_file)
        
        # ==========================================
        # 1. PD Processing (Absolute Polarization Difference)
        # ==========================================
        pd_img = np.abs(s_img - p_img)
        pd_scaled = np.clip(pd_img / (pd_limit + 1e-12), 0.0, 1.0)
        pd_16 = np.round(pd_scaled * 65535).astype(np.uint16)
        
        label_pd_dir = os.path.join(img_out_dirs["PD"], label)
        os.makedirs(label_pd_dir, exist_ok=True)
        pd_save_path = os.path.join(label_pd_dir, base_name)
        tifffile.imwrite(pd_save_path, pd_16)
        
        # PD GLCM feature extraction (using INTER_AREA interpolation)
        pd_resized = cv2.resize(
            pd_scaled,
            (resize_size, resize_size),
            interpolation=cv2.INTER_AREA
        )
        pd_feats = extract_glcm_features(pd_resized)
        
        # ==========================================
        # 2. NPD Processing (Normalized Polarization Difference)
        # ==========================================
        npd_img = (s_img - p_img) / (s_img + p_img + 1e-8)
        vmin, vmax = 0.15, 1.00
        npd_norm = np.clip((npd_img - vmin) / (vmax - vmin + 1e-8), 0.0, 1.0)
        
        gamma = 0.7  # Non-linear Gamma correction to suppress overexposure/washout
        npd_norm = np.power(npd_norm, gamma)
        npd_16 = np.round(npd_norm * 65535).astype(np.uint16)
        
        label_npd_dir = os.path.join(img_out_dirs["NPD"], label)
        os.makedirs(label_npd_dir, exist_ok=True)
        npd_save_path = os.path.join(label_npd_dir, base_name)
        tifffile.imwrite(npd_save_path, npd_16)
        
        # NPD GLCM feature extraction (using INTER_AREA interpolation)
        npd_resized = cv2.resize(
            npd_norm,
            (resize_size, resize_size),
            interpolation=cv2.INTER_AREA
        )
        npd_feats = extract_glcm_features(npd_resized)
        
        return label, pd_save_path, pd_feats, npd_save_path, npd_feats
    except Exception as e:
        print(f"Error processing {p_file}: {e}")
        return None

def evaluate_classifiers(X, y, name=""):
    """Evaluate performance across multiple models (SVM multi-kernel, Random Forest) with strict split-then-normalize"""
    print(f"\n--- [{name}] Comprehensive Multi-Model Classification Evaluation ---")
    
    # 1. 8:1:1 split (first split out 10% test set)
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y, test_size=0.1, random_state=random_seed, stratify=y
    )
    # Then split the remainder into training and validation sets (~8:1 ratio)
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=0.1111, random_state=random_seed, stratify=y_train_val
    )
    
    # Strict anti-leakage: split datasets first, fit scaler on training set only, then transform val/test separately
    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    
    # Define models to evaluate
    models = {
        "SVM-linear": SVC(kernel='linear', random_state=random_seed),
        "SVM-rbf": SVC(kernel='rbf', random_state=random_seed),
        "SVM-poly": SVC(kernel='poly', random_state=random_seed),
        "SVM-sigmoid": SVC(kernel='sigmoid', random_state=random_seed),
        "Random Forest": RandomForestClassifier(random_state=random_seed, n_estimators=100)
    }
    
    for model_name, clf in models.items():
        # 2. Independent validation set and test set evaluation
        clf.fit(X_train_scaled, y_train)
        val_acc = accuracy_score(y_val, clf.predict(X_val_scaled))
        test_acc = accuracy_score(y_test, clf.predict(X_test_scaled))
        
        # 3. 5-fold cross-validation (strict fold-wise independent normalization on Train+Val dataset)
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_seed)
        cv_accuracies = []
        for train_idx, val_idx in skf.split(X_train_val, y_train_val):
            X_tr, X_v = X_train_val[train_idx], X_train_val[val_idx]
            y_tr, y_v = y_train_val[train_idx], y_train_val[val_idx]
            
            fold_scaler = RobustScaler()
            X_tr_scaled = fold_scaler.fit_transform(X_tr)
            X_v_scaled = fold_scaler.transform(X_v)
            
            # Re-instantiate model
            if model_name == "SVM-linear":
                fold_clf = SVC(kernel='linear', random_state=random_seed)
            elif model_name == "SVM-rbf":
                fold_clf = SVC(kernel='rbf', random_state=random_seed)
            elif model_name == "SVM-poly":
                fold_clf = SVC(kernel='poly', random_state=random_seed)
            elif model_name == "SVM-sigmoid":
                fold_clf = SVC(kernel='sigmoid', random_state=random_seed)
            else:
                fold_clf = RandomForestClassifier(random_state=random_seed, n_estimators=100)
                
            fold_clf.fit(X_tr_scaled, y_tr)
            cv_accuracies.append(accuracy_score(y_v, fold_clf.predict(X_v_scaled)))
            
        print(f"  👉 [{model_name}]")
        print(f"     Val Acc: {val_acc * 100:.2f}% | Test Acc: {test_acc * 100:.2f}% | 5-Fold CV Acc: {np.mean(cv_accuracies) * 100:.2f}% (±{np.std(cv_accuracies) * 100:.2f}%)")

# -----------------------------
# 3. Collect All Image Pairs
# -----------------------------
np.random.seed(random_seed)
selected_pairs = []

for label, folder in data_dirs.items():
    pairs = get_image_pairs(folder)
    print(f"Found {len(pairs)} available image pairs in folder [{label}]")
    for p_file, s_file in pairs:
        selected_pairs.append((p_file, s_file, label))

print(f"\nTotal number of image pairs collected for processing: {len(selected_pairs)}")

# ==========================================
# 4. Estimate pd_limit using training samples only (strict anti-leakage)
# ==========================================
pair_labels = np.array([item[2] for item in selected_pairs])

train_pairs, _ = train_test_split(
    selected_pairs,
    test_size=0.1,
    random_state=random_seed,
    stratify=pair_labels
)

print("\nEstimating PD scaling upper limit (pd_limit) based on training samples...")
pd_percentiles = []
for p_file, s_file, label in tqdm(train_pairs, desc="Estimating pd_limit"):
    p_img = tifffile.imread(p_file).astype(np.float32) / 65535.0
    s_img = tifffile.imread(s_file).astype(np.float32) / 65535.0
    pd_img = np.abs(s_img - p_img)
    valid_pixels = pd_img[np.isfinite(pd_img)]
    if valid_pixels.size > 0:
        p99 = np.percentile(valid_pixels, 99)
        pd_percentiles.append(p99)

pd_limit = np.percentile(pd_percentiles, 95)
print(f"Estimated PD limit: {pd_limit:.6f}")
print(f"Number of training samples used for limit estimation: {len(train_pairs)}")

# ==========================================
# 5. Multithreading Parallel Processing of All Image Pairs and GLCM Extraction
# ==========================================
pd_results = []
npd_results = []

print("\nStarting multithreading parallel processing of all image pairs, image saving, and GLCM feature extraction...")
max_workers = min(16, (os.cpu_count() or 1) * 2)

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = [executor.submit(process_single_pair, item, pd_limit) for item in selected_pairs]
    
    for future in tqdm(as_completed(futures), total=len(futures), desc="Processing Pairs"):
        res = future.result()
        if res is not None:
            label, pd_path, pd_feats, npd_path, npd_feats = res
            pd_results.append([label, pd_path] + pd_feats)
            npd_results.append([label, npd_path] + npd_feats)

# -----------------------------
# 6. Generate Descriptive Feature Names and Save Raw Feature CSV Files
# -----------------------------
pd_feature_columns = generate_feature_names("PD")
npd_feature_columns = generate_feature_names("NPD")

df_pd = pd.DataFrame(pd_results, columns=['Label', 'Path'] + pd_feature_columns)
df_npd = pd.DataFrame(npd_results, columns=['Label', 'Path'] + npd_feature_columns)

pd_csv_path = os.path.join(csv_out_dir, "pd_glcm_features_raw.csv")
df_pd.to_csv(pd_csv_path, index=False)

npd_csv_path = os.path.join(csv_out_dir, "npd_glcm_features_raw.csv")
df_npd.to_csv(npd_csv_path, index=False)

print(f"\n✅ All raw feature CSVs with descriptive feature columns have been successfully saved to: {csv_out_dir}")

# -----------------------------
# 7. Comprehensive Multi-Model Classification Performance Evaluation
# -----------------------------
X_pd = df_pd[pd_feature_columns].values
y_pd = df_pd['Label'].values

X_npd = df_npd[npd_feature_columns].values
y_npd = df_npd['Label'].values

evaluate_classifiers(X_pd, y_pd, name="PD (|S-P|) Features [All Samples]")
evaluate_classifiers(X_npd, y_npd, name="NPD ((S-P)/(S+P)) Features [All Samples]")

print("\n🎉 All processes completed successfully!")

In [ ]:
search for C value

In [ ]:
import os
import multiprocessing
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import RobustScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

# -----------------------------
# 1. Configuration of Paths and Hyperparameters
# -----------------------------
csv_dir = r"E:\0. Research\1. Paper_data\2. Hacat cisplatin ROS\Original\差分TestAll\CSV"
output_report_dir = r"E:\0. Research\1. Paper_data\2. Hacat cisplatin ROS\Original\差分TestAll\GridSearch_Report"
os.makedirs(output_report_dir, exist_ok=True)

pd_csv_path = os.path.join(csv_dir, "pd_glcm_features_raw.csv")
npd_csv_path = os.path.join(csv_dir, "npd_glcm_features_raw.csv")

# Specified list of C values
C_values = [0.001, 0.1, 1, 5, 10, 50, 100, 150, 200]

# 3 different random seeds
seeds = [42, 123, 2026]
n_splits = 5

feature_columns = [f'feat_{i}' for i in range(140)]

# Automatically get available CPU core count (reserve 1 core to prevent system lag)
num_cores = max(1, multiprocessing.cpu_count() - 1)

# -----------------------------
# 2. Evaluation function for a single C value (for multiprocessing parallelization)
# -----------------------------
def evaluate_single_C(C, X, y, seeds, n_splits):
    """Evaluate all seeds and 5-fold cross-validation for a single C value"""
    fold_accuracies = []
    
    for seed in seeds:
        # 1. 8:1:1 split (first split out 10% test set)
        X_train_val, X_test, y_train_val, y_test = train_test_split(
            X, y, test_size=0.1, random_state=seed, stratify=y
        )
        
        # 2. 5-fold cross-validation (strict fold-wise independent normalization)
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for train_idx, val_idx in skf.split(X_train_val, y_train_val):
            X_tr, X_v = X_train_val[train_idx], X_train_val[val_idx]
            y_tr, y_v = y_train_val[train_idx], y_train_val[val_idx]
            
            # Strict anti-leakage: fit the scaler using only the current fold's training set
            scaler = RobustScaler()
            X_tr_scaled = scaler.fit_transform(X_tr)
            X_v_scaled = scaler.transform(X_v)
            
            # Use LinearSVC to speed up linear kernel computation
            clf = LinearSVC(C=C, random_state=seed, dual="auto", max_iter=10000)
            clf.fit(X_tr_scaled, y_tr)
            
            acc = accuracy_score(y_v, clf.predict(X_v_scaled))
            fold_accuracies.append(acc * 100)  # Convert to percentage
            
    mean_acc = np.mean(fold_accuracies)
    std_acc = np.std(fold_accuracies)
    
    print(f"  👉 C = {C:<6} | Mean Acc: {mean_acc:.2f}%  ±  SD: {std_acc:.2f}%")
    
    return {
        "C_Value": C,
        "Mean_Accuracy(%)": round(mean_acc, 2),
        "Std_Dev(%)": round(std_acc, 2)
    }

# -----------------------------
# 3. Grid search master control function (integrated with multiprocessing concurrency)
# -----------------------------
def run_grid_search_svm(csv_path, feature_name="PD"):
    print(f"\n==============================================")
    print(f"Starting multiprocessing SVM-linear grid search for [{feature_name}] features...")
    print(f"==============================================")
    
    if not os.path.exists(csv_path):
        print(f"❌ File not found: {csv_path}")
        return
        
    df = pd.read_csv(csv_path)
    X = df[feature_columns].values
    y = df['Label'].values
    
    # Use Joblib to compute grid search for all C values in parallel
    results = Parallel(n_jobs=num_cores)(
        delayed(evaluate_single_C)(C, X, y, seeds, n_splits) for C in C_values
    )
    
    # Reorder results by C value size
    results = sorted(results, key=lambda x: x["C_Value"])
    
    # Convert to DataFrame and save report
    df_res = pd.DataFrame(results)
    out_csv = os.path.join(output_report_dir, f"{feature_name}_svm_linear_grid_search.csv")
    df_res.to_csv(out_csv, index=False)
    
    # Print the full summary table in the console for easy review of each C value's performance
    print(f"\n📊 [{feature_name}] Grid Search Summary Table for Each C Value:")
    print(df_res.to_string(index=False))
    print(f"\n✅ [{feature_name}] Grid search results successfully saved to: {out_csv}")
    
    # Print the optimal C value
    best_row = df_res.loc[df_res['Mean_Accuracy(%)'].idxmax()]
    print(f"🏆 [{feature_name}] Optimal parameters: C = {best_row['C_Value']} (Accuracy: {best_row['Mean_Accuracy(%)']}% ± {best_row['Std_Dev(%)']}%)")

# -----------------------------
# 4. Execute parallel grid search for PD and NPD features
# -----------------------------
if __name__ == '__main__':
    print(f"💻 Multiprocessing acceleration enabled, parallel working core count: {num_cores}")
    run_grid_search_svm(pd_csv_path, feature_name="PD")
    run_grid_search_svm(npd_csv_path, feature_name="NPD")

    print("\n🎉 All grid search and evaluation processes completed successfully!")

In [ ]:
C=5 performance

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    accuracy_score
)

# -----------------------------
# 1. Configuration of Paths and Hyperparameters
# -----------------------------
csv_dir = r"E:\0. Research\1. Paper_data\2. Hacat cisplatin ROS\Original\差分TestAll\CSV"
output_report_dir = r"E:\0. Research\1. Paper_data\2. Hacat cisplatin ROS\Original\差分TestAll\Evaluation_C5"
os.makedirs(output_report_dir, exist_ok=True)

pd_csv_path = os.path.join(csv_dir, "pd_glcm_features_raw.csv")
npd_csv_path = os.path.join(csv_dir, "npd_glcm_features_raw.csv")

C_optimal = 5
seeds = [42, 123, 2026]  # 3 random seeds
n_splits = 3             # 3-fold cross-validation

# -----------------------------
# 2. Core Evaluation and Plotting Function
# -----------------------------
def evaluate_and_plot_feature_set(csv_path, feature_name):
    print(f"\n==============================================")
    print(f" Evaluating [{feature_name}] features (SVM-linear, C = {C_optimal})...")
    print(f"==============================================")
    
    if not os.path.exists(csv_path):
        print(f"❌ File not found: {csv_path}")
        return
        
    df = pd.read_csv(csv_path)
    
    # Dynamically retrieve feature columns (automatically excluding metadata columns 'Label', 'Path')
    metadata_cols = ['Label', 'Path']
    feature_columns = [col for col in df.columns if col not in metadata_cols]
    print(f"ℹ️ Automatically recognized [{feature_name}] feature dimension: {len(feature_columns)} dimensions")
    
    X = df[feature_columns].values
    y = df['Label'].values
    classes = np.unique(y)
    
    feature_output_dir = os.path.join(output_report_dir, feature_name)
    os.makedirs(feature_output_dir, exist_ok=True)

    all_y_true = []
    all_y_pred = []

    # Multi-seed and multi-fold cross-validation loop
    for seed in seeds:
        # 8:1:1 split
        X_train_val, X_test, y_train_val, y_test = train_test_split(
            X, y, test_size=0.1, random_state=seed, stratify=y
        )
        
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for train_idx, val_idx in skf.split(X_train_val, y_train_val):
            X_tr, X_v = X_train_val[train_idx], X_train_val[val_idx]
            y_tr, y_v = y_train_val[train_idx], y_train_val[val_idx]
            
            # Strict fold-wise independent normalization to prevent data leakage
            scaler = RobustScaler()
            X_tr_scaled = scaler.fit_transform(X_tr)
            X_v_scaled = scaler.transform(X_v)
            
            # Train SVC linear model with optimal C
            clf = SVC(C=C_optimal, kernel='linear', random_state=seed)
            clf.fit(X_tr_scaled, y_tr)
            
            preds = clf.predict(X_v_scaled)
            all_y_true.extend(y_v)
            all_y_pred.extend(preds)

    all_y_true = np.array(all_y_true)
    all_y_pred = np.array(all_y_pred)

    # 3. Compute evaluation metrics (raw numeric form kept to four decimal places)
    overall_accuracy = accuracy_score(all_y_true, all_y_pred)
    report_dict = classification_report(all_y_true, all_y_pred, target_names=classes, output_dict=True)

    print(f"\n[{feature_name}] Overall Accuracy: {overall_accuracy:.4f}\n")
    print(f"{'Class':<12} | {'Precision':<10} | {'Recall':<10} | {'F1-Score':<10}")
    print("-" * 50)
    for cls in classes:
        p = report_dict[cls]['precision']
        r = report_dict[cls]['recall']
        f = report_dict[cls]['f1-score']
        print(f"{cls:<12} | {p:<10.4f} | {r:<10.4f} | {f:<10.4f}")

    print("-" * 50)
    print(f"{'Macro Avg':<12} | {report_dict['macro avg']['precision']:<10.4f} | {report_dict['macro avg']['recall']:<10.4f} | {report_dict['macro avg']['f1-score']:<10.4f}")
    print(f"{'Weighted Avg':<12} | {report_dict['weighted avg']['precision']:<10.4f} | {report_dict['weighted avg']['recall']:<10.4f} | {report_dict['weighted avg']['f1-score']:<10.4f}")

    # Save classification report CSV
    metrics_df = pd.DataFrame(report_dict).transpose()
    metrics_report_path = os.path.join(feature_output_dir, f"{feature_name}_classification_report_C{C_optimal}.csv")
    metrics_df.to_csv(metrics_report_path)

    # 4. Plot and save confusion matrix
    cm = confusion_matrix(all_y_true, all_y_pred, labels=classes)
    plt.figure(figsize=(7, 6), dpi=300)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes, cbar=True)
    plt.title(f'[{feature_name}] Confusion Matrix (SVM-linear, C={C_optimal})', fontsize=12, fontweight='bold')
    plt.xlabel('Predicted Label', fontsize=11)
    plt.ylabel('True Label', fontsize=11)
    plt.tight_layout()
    cm_path = os.path.join(feature_output_dir, f"{feature_name}_confusion_matrix.png")
    plt.savefig(cm_path, dpi=300)
    plt.close()

    # 5. Compute and plot pairwise accuracy matrix (OvR approach)
    n_classes = len(classes)
    pairwise_acc_matrix = np.zeros((n_classes, n_classes))

    for i in range(n_classes):
        for j in range(n_classes):
            if i == j:
                pairwise_acc_matrix[i, j] = 1.0
            else:
                mask = (all_y_true == classes[i]) | (all_y_true == classes[j])
                sub_true = all_y_true[mask]
                sub_pred = all_y_pred[mask]
                if len(sub_true) > 0:
                    pairwise_acc_matrix[i, j] = accuracy_score(sub_true, sub_pred)

    pairwise_df = pd.DataFrame(pairwise_acc_matrix, index=classes, columns=classes)
    print(f"\n--- [{feature_name}] Pairwise Accuracy Matrix (OvR) ---")
    print(pairwise_df.round(4).to_string())
    
    pairwise_csv_path = os.path.join(feature_output_dir, f"{feature_name}_pairwise_accuracy_matrix.csv")
    pairwise_df.to_csv(pairwise_csv_path)

    # Plot pairwise accuracy matrix heatmap
    plt.figure(figsize=(7, 6), dpi=300)
    sns.heatmap(pairwise_df, annot=True, fmt='.4f', cmap='Purples', vmin=0, vmax=1, xticklabels=classes, yticklabels=classes, cbar=True)
    plt.title(f'[{feature_name}] Pairwise Accuracy Matrix (C={C_optimal})', fontsize=12, fontweight='bold')
    plt.xlabel('Target Class', fontsize=11)
    plt.ylabel('Reference Class', fontsize=11)
    plt.tight_layout()
    pairwise_plot_path = os.path.join(feature_output_dir, f"{feature_name}_pairwise_accuracy_matrix.png")
    plt.savefig(pairwise_plot_path, dpi=300)
    plt.close()
    
    print(f"\n✅ All reports and plots for [{feature_name}] have been successfully saved to: {feature_output_dir}")

# -----------------------------
# 3. Main Program Entry
# -----------------------------
if __name__ == '__main__':
    evaluate_and_plot_feature_set(pd_csv_path, feature_name="PD")
    evaluate_and_plot_feature_set(npd_csv_path, feature_name="NPD")
    print("\n🎉 Detailed metrics evaluation and plotting tasks for PD and NPD features at C=5 completed successfully!")